# Chapter 5 — Library and reusable Composite

Engineer course · source candidate · CONVERGING

# Chapter 5 — Library and reusable Composite

This complete Chapter is one clean-kernel execution unit. Its three web
Lessons are reading views over the same ordered source fragments. QMD is
the editable authority; `chapter.ipynb` is a generated zero-output
transport artifact.

## Lesson 1 — Put one physical unit behind a Library boundary

### Define the catalog and factory

Use a Composite when a group of elements has one owner, a deliberate
public boundary, and a useful name outside its implementation. This
grounded LC is a good reusable unit: the factory owns its capacitor,
inductor, local bus, and ground relation. The parent will own the
coupler and Port.

In [ ]:
from IPython.display import display

from scnsim import (
    CircuitDiagramSpec,
    CircuitPlan,
    CompositePlan,
    Library,
    ParameterDefinitions,
    ParameterRef,
    ParameterSpec,
    Theme,
    components,
    units as u,
)


class ResonatorLibrary(Library):
    """Project-owned reusable resonator catalog."""

    def parallel_linear_lc_resonator(self, *, id, capacitance, inductance):
        """Build one grounded LC with a deliberate public boundary."""
        composite = CompositePlan(id=id, library=self)
        capacitor = composite.add(
            components.capacitor(id="capacitor", capacitance=capacitance)
        )
        inductor = composite.add(
            components.inductor(id="inductor", inductance=inductance)
        )
        if isinstance(capacitance, ParameterRef):
            composite.expose_parameter(id="capacitance", parameter=capacitance)
        if isinstance(inductance, ParameterRef):
            composite.expose_parameter(id="inductance", parameter=inductance)
        terminal_bus = composite.bus(id="terminal")
        composite.parallel(
            id="parallel_lc",
            start=terminal_bus,
            branches=((capacitor,), (inductor,)),
            end=composite.ground,
        )
        composite.expose_pin(id="terminal", at=terminal_bus)
        composite.expose_pin(id="alternate_terminal", at=terminal_bus)
        composite.expose_coordinate(id="terminal_node", at=terminal_bus)
        return composite.build()


components_library = ResonatorLibrary()

The two Pins are equivalent public wiring occurrences on the same
internal signal bus. The Coordinate is analysis access to that bus, not
a wire. C and L are published only when the caller supplied real
`ParameterRef` values.

## Lesson 2 — Instantiate and inspect the public surface

### Bind C and L through physical fields

The factory receives the two independent public inputs. Their baseline
values match the earlier grounded-LC Chapters.

In [ ]:
inputs = ParameterDefinitions(id="readout_design")
capacitance = inputs.parameter(
    id="capacitance",
    baseline=110.0 * u.fF,
    spec=ParameterSpec(unit=u.fF),
)
inductance = inputs.parameter(
    id="inductance",
    baseline=5.8 * u.nH,
    spec=ParameterSpec(unit=u.nH),
)

plan = CircuitPlan(id="reusable_resonator")
resonator = plan.add(
    components_library.parallel_linear_lc_resonator(
        id="resonator",
        capacitance=capacitance,
        inductance=inductance,
    )
)
resonator_terminal = resonator.pin("terminal")
resonator_alternate_terminal = resonator.pin("alternate_terminal")
resonator_coordinate = resonator.coordinate("terminal_node")
resonator_capacitance = resonator.parameter("capacitance")
resonator_inductance = resonator.parameter("inductance")

The parent receives only those five handles. It cannot reach the
factory’s private capacitor, inductor, bus, or ground structure.

In [ ]:
display(
    {
        "terminal Pin": resonator_terminal,
        "equivalent open Pin": resonator_alternate_terminal,
        "analysis Coordinate": resonator_coordinate,
        "capacitance Parameter": resonator_capacitance,
        "inductance Parameter": resonator_inductance,
    }
)

`alternate_terminal` is deliberately left externally open. An open
published boundary is still part of the component interface; it needs no
fake load or analysis-only root wiring.

## Lesson 3 — Reuse the component from its boundary

### Couple the parent directly to the child Pin

The 6 fF coupler and terminated 50 ohm Port belong to the parent. The
series relation ends at the child’s public Pin; the Coordinate does not
require a second parent Bus.

In [ ]:
signal_bus = plan.bus(id="signal_boundary")
coupler = plan.add(
    components.capacitor(id="coupling_cap", capacitance=6.0 * u.fF)
)
plan.series(
    id="coupling",
    start=signal_bus,
    elements=(coupler,),
    end=resonator_terminal,
)
signal_port = plan.add_port(
    id="signal_in",
    at=signal_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

### Review the authored parent/child boundary

In [ ]:
reused_diagram = plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        theme=Theme.AUTO,
        show_parameter_values=True,
        show_provenance=True,
    )
)
reused_diagram.show()

In [ ]:
reused_diagram.audit.show()

The authoring certificate and audit describe the exact Plan boundary.
They do not publish private editable handles from the built Composite.